# 12b – Data Imputation, Encoding & Feature Engineering


**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

## Table of Contents

1. **Introduction & Goals**
2. **Imports & Paths**
3. **Load Processed Data**
4. **Train / Validation / Test Split**
5. **Missing Value Overview**
6. **Define Feature Types (numeric vs. categorical)**
7. **Imputation**
   - 7.1 Simple median/mode imputation (leakage-free)
   - 7.2 Optional RF-based imputation for 1 useful column
8. **Encoding**
   - 8.1 Low-cardinality → One-Hot
   - 8.2 High-cardinality → keep for later steps
9. **Feature Engineering**
10. **(Optional) Scaling**
11. **Save Final Datasets**
12. **Notes for the Report**


In [29]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

# import onehotencoder
from sklearn.preprocessing import OneHotEncoder

# import scaler = RobustScaler()
from sklearn.preprocessing import RobustScaler

RANDOM_STATE = 42


## 1. Load processed data

In [15]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df_train = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
df_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# Drop the column carID from both dataframes because it is not needed for modeling
if "carID" in df_train.columns:
    df_train = df_train.drop(columns=["carID"])
if "carID" in df_test.columns:
    df_test = df_test.drop(columns=["carID"])
    

# Separate features and target variable from training data
print("Loaded shape:", df_train.shape)
display(df_train.head(3))

print("Loaded test shape:", df_test.shape)
display(df_test.head(3))


Loaded shape: (75973, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 12)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


## 2. Train / validation / test split

In [16]:
TARGET_COL = "price"

if "split" in df_train.columns:
    train_mask = df_train["split"].eq("train")
    val_mask   = df_train["split"].eq("val")
else:
    train_mask = np.random.rand(len(df_train)) < 0.8
    val_mask   = ~train_mask

train_df = df_train.loc[train_mask].copy()
val_df   = df_train.loc[val_mask].copy()

y_train = train_df[TARGET_COL].copy()
y_val   = val_df[TARGET_COL].copy()

drop_cols = [TARGET_COL, "split"]

X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
X_val   = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
X_test  = df_test.copy()

X_train.shape, X_val.shape, X_test.shape


((60835, 12), (15138, 12), (32567, 12))

## 3. Missing value overview

In [17]:
def missing_report(df, name):
    miss = df.isna().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    print(f"--- {name} ---")
    if miss.empty:
        print("no missing values")
    else:
        print(miss)

missing_report(X_train, "X_train")
missing_report(X_val, "X_val")
missing_report(X_test, "X_test")


--- X_train ---
mpg               0.121772
tax               0.110397
engineSize        0.027681
mileage           0.024377
fuelType          0.020597
previousOwners    0.020465
transmission      0.020383
paintQuality%     0.020383
hasDamage         0.020136
model             0.019939
Brand             0.019824
year              0.019709
dtype: float64
--- X_val ---
mpg               0.118378
tax               0.103448
engineSize        0.026886
mileage           0.023055
hasDamage         0.021337
Brand             0.020809
fuelType          0.020412
previousOwners    0.020148
model             0.020082
transmission      0.019686
year              0.019289
paintQuality%     0.018761
dtype: float64
--- X_test ---
tax               0.101575
mpg               0.100961
mileage           0.021156
fuelType          0.020143
year              0.020051
model             0.019959
Brand             0.019928
engineSize        0.019283
paintQuality%     0.019191
transmission      0.019130
previou

## 4. Define feature types

In [19]:
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_cols[:10], categorical_cols[:10]


(['year',
  'mileage',
  'tax',
  'mpg',
  'engineSize',
  'paintQuality%',
  'previousOwners',
  'hasDamage'],
 ['Brand', 'model', 'transmission', 'fuelType'])

## 5. Simple imputations (median/mode)

In [ ]:
from sklearn.impute import SimpleImputer

# 1) Numeric
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[numeric_cols])  # only train data to prevent data leakage

X_train_imp = X_train.copy()
X_val_imp   = X_val.copy()
X_test_imp  = X_test.copy()

X_train_imp[numeric_cols] = num_imputer.transform(X_train[numeric_cols])
X_val_imp[numeric_cols]   = num_imputer.transform(X_val[numeric_cols])
X_test_imp[numeric_cols]  = num_imputer.transform(X_test[numeric_cols])

# 2) Categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train[categorical_cols])  # only train data to prevent data leakage

X_train_imp[categorical_cols] = cat_imputer.transform(X_train[categorical_cols])
X_val_imp[categorical_cols]   = cat_imputer.transform(X_val[categorical_cols])
X_test_imp[categorical_cols]  = cat_imputer.transform(X_test[categorical_cols])

missing_report(X_train_imp, "X_train_imp")
missing_report(X_val_imp, "X_val_imp")
missing_report(X_test_imp, "X_test_imp")


### 5.1 Optional: RF imputation for one useful column

In [22]:
col_to_impute = None
for cand in ["mileage", "Mileage", "km"]:
    if cand in X_train.columns and X_train[cand].isna().any():
        col_to_impute = cand
        break

if col_to_impute is not None:
    feat_cols = [c for c in X_train_imp.columns if c != col_to_impute]

    train_rows = X_train[col_to_impute].notna()
    X_rf = pd.get_dummies(X_train_imp.loc[train_rows, feat_cols], drop_first=True)
    y_rf = X_train.loc[train_rows, col_to_impute]

    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_rf, y_rf)

    # train
    missing_train = X_train_imp[col_to_impute].isna()
    if missing_train.any():
        X_missing = pd.get_dummies(X_train_imp.loc[missing_train, feat_cols], drop_first=True)
        X_missing = X_missing.reindex(columns=X_rf.columns, fill_value=0)
        X_train_imp.loc[missing_train, col_to_impute] = rf.predict(X_missing)

    # val
    missing_val = X_val_imp[col_to_impute].isna()
    if missing_val.any():
        X_missing = pd.get_dummies(X_val_imp.loc[missing_val, feat_cols], drop_first=True)
        X_missing = X_missing.reindex(columns=X_rf.columns, fill_value=0)
        X_val_imp.loc[missing_val, col_to_impute] = rf.predict(X_missing)

    # test
    missing_test = X_test_imp[col_to_impute].isna()
    if missing_test.any():
        X_missing = pd.get_dummies(X_test_imp.loc[missing_test, feat_cols], drop_first=True)
        X_missing = X_missing.reindex(columns=X_rf.columns, fill_value=0)
        X_test_imp.loc[missing_test, col_to_impute] = rf.predict(X_missing)


## 6. Encoding

In [24]:
low_card_cat = [c for c in categorical_cols if X_train_imp[c].nunique() <= 15]
high_card_cat = [c for c in categorical_cols if c not in low_card_cat]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train_imp[low_card_cat])

def apply_ohe(df):
    ohe_arr = ohe.transform(df[low_card_cat])
    ohe_cols = ohe.get_feature_names_out(low_card_cat)
    ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
    df_rest = df.drop(columns=low_card_cat)
    return pd.concat([df_rest, ohe_df], axis=1)

X_train_enc = apply_ohe(X_train_imp)
X_val_enc   = apply_ohe(X_val_imp)
X_test_enc  = apply_ohe(X_test_imp)


# print the columns after encoding
print("Columns after encoding:")
print(X_test_enc.columns.tolist())


# Print the number of columns after encoding for test val and train
print("Number of columns after encoding:")
print("X_train_enc:", X_train_enc.shape[1])
print("X_val_enc:", X_val_enc.shape[1])
print("X_test_enc:", X_test_enc.shape[1])

Columns after encoding:
['model', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'Brand_Audi', 'Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes-Benz', 'Brand_Opel', 'Brand_Toyota', 'Brand_Volkswagen', 'Brand_Škoda', 'transmission_Automatic', 'transmission_Manual', 'transmission_Semi-Auto', 'transmission_Unknown', 'fuelType_Diesel', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol']
Number of columns after encoding:
X_train_enc: 27
X_val_enc: 27
X_test_enc: 27


## 7. Feature engineering


In [26]:
CURRENT_YEAR = 2025

def add_features(df, target=None):
    df = df.copy()

    if "year" in df.columns:
        df["car_age"] = CURRENT_YEAR - df["year"]
        df.loc[df["car_age"] < 0, "car_age"] = np.nan

    if "mileage" in df.columns and "car_age" in df.columns:
        df["mileage_per_year"] = df["mileage"] / df["car_age"]
        df["mileage_per_year"] = df["mileage_per_year"].replace([np.inf, -np.inf], np.nan)

    return df

X_train_fe = add_features(X_train_enc)
X_val_fe   = add_features(X_val_enc)
X_test_fe  = add_features(X_test_enc, target=None)

X_train_fe.shape, X_val_fe.shape, X_test_fe.shape


((60835, 29), (15138, 29), (32567, 29))

## 8. Scaling

We applied a RobustScaler to the numerical features because the dataset contains outliers (price, mileage, car_age). RobustScaler uses the median and the interquartile range instead of mean and standard deviation, which makes the scaling more robust and stable. The scaler was fitted on the training set only and then applied to validation and test to avoid data leakage.”

In [30]:
scaler = RobustScaler()

# numeric cols after FE (on train)
num_after_enc = X_train_fe.select_dtypes(include=["number"]).columns

# fit on TRAIN only
scaler.fit(X_train_fe[num_after_enc])

# copy
X_train_final = X_train_fe.copy()
X_val_final   = X_val_fe.copy()
X_test_final  = X_test_fe.copy()

# scale train and val on same cols
X_train_final[num_after_enc] = scaler.transform(X_train_fe[num_after_enc])
X_val_final[num_after_enc]   = scaler.transform(X_val_fe[num_after_enc])

# for test: only the intersection of cols
test_cols = [c for c in num_after_enc if c in X_test_fe.columns]
X_test_final[test_cols] = scaler.transform(X_test_fe[test_cols])


In [31]:
# Check for NaN values in each dataset
def check_nan(df, name):
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        print(f"{name} has NaN values in columns: {nan_cols}")
    else:
        print(f"{name} has no NaN values.")


check_nan(X_train_final, "X_train_final")
check_nan(y_train.to_frame(), "y_train")
check_nan(X_val_final, "X_val_final")
check_nan(y_val.to_frame(), "y_val")
check_nan(X_test_final, "X_test_final")

X_train_final has no NaN values.
y_train has no NaN values.
X_val_final has no NaN values.
y_val has no NaN values.
X_test_final has no NaN values.


## 9. Save

In [33]:
# Save Processed Datasets
"""
Save X_train, y_train, X_val, y_val, and X_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
X_train_final.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

X_val_final.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

X_test_final.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("Processed data saved successfully (X/y separated):")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}")

Processed data saved successfully (X/y separated):
X_train: (60835, 12), y_train: (60835,)
X_val:   (15138, 12), y_val: (15138,)
X_test:  (32567, 12)
